<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/Orbit_Viewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
%pip install pymcel -Uq

In [12]:
import numpy as np
import plotly.graph_objects as go
from dataclasses import dataclass
import pymcel as pc
import spiceypy as spy
import matplotlib.pyplot as plt

# Visualizador de órbitas 3D aplicando las 3 rotaciones de Euler con elementos orbitales

## Rutina para aplicar las rotaciones

La siguiente función `orbital_elements_to_orbit_3d` toma los elementos orbitales clásicos (excentricidad, semieje mayor, inclinación, longitud del nodo ascendente y argumento del periapsis) y genera las coordenadas 3D de la órbita en el sistema de referencia astronómico. Utiliza las tres rotaciones de Euler para orientar la órbita correctamente en el espacio.


In [13]:
def orbital_elements_to_orbit_3d(e, a, i, node, peri, num_points=100):
    """
    Convierte los elementos orbitales clásicos en una órbita 3D en el espacio.

    Args:
        e (float): Excentricidad.
        a (float): Semieje mayor.
        i (float): Inclinación (en radianes).
        node (float): Longitud del nodo ascendente (en radianes).
        peri (float): Argumento del periapsis (en radianes).
        num_points (int): Número de puntos para generar la órbita.

    Returns:
        np.ndarray: Array de Nx3 con las coordenadas (x, y, z) de la órbita.
    """

    # Calcular el parámetro semilato recto (p)
    p = a * (1 - e**2) if e < 1 else a * (e**2 - 1) # Asegurar que 'p' es positivo para hiperbolas
    if e == 1: # Caso parabolico
        p = 2 * a # a es la distancia al periapsis para una parabola

    # Generar ángulos de anomalía verdadera (f) para dibujar la órbita
    # Si es una órbita hiperbólica, los límites de f son diferentes.
    if e < 1: # Elipse
        fs = np.linspace(0, 2 * np.pi, num_points)
    elif e == 1: # Parábola, se necesita un rango adecuado
        # Para una parábola, la anomalía verdadera puede ir de -pi a pi, pero la órbita es infinita.
        # Usaremos un rango que muestre una buena parte de la trayectoria.
        f_limit = np.arccos(-1/e + 0.001) # Límite para evitar division por cero
        fs = np.linspace(-f_limit, f_limit, num_points) # Valores cercanos al periapsis
    else: # Hipérbola
        # Los límites de la anomalía verdadera para una hipérbola
        f_limit = np.arccos(-1/e + 0.001) # Pequeña perturbación para evitar arccos(-1)
        fs = np.linspace(-f_limit + 0.1, f_limit - 0.1, num_points) # Evitar los asíntotas

    # Calcular la distancia radial (r) para cada f
    rs = p / (1 + e * np.cos(fs))

    # Coordenadas en el plano perifocal (xy) con z=0
    xfs = rs * np.cos(fs)
    yfs = rs * np.sin(fs)
    zfs = np.zeros_like(xfs)

    # Convertir a array de 3xN para multiplicación matricial
    r_perifocal = np.array([xfs, yfs, zfs])

    # Matrices de rotación (del sistema perifocal al astronómico)
    # Nota: spiceypy.rotate(angle, axis) devuelve la matriz de rotación CONTRARIA
    # a la que se usa para transformar un vector, si se usa como M @ v.
    # Para pasar del sistema perifocal al astronómico, las rotaciones son las inversas.
    # O lo que es lo mismo, rotar por ángulos negativos en el orden inverso.

    # Rotaciones de Euler (Z-X-Z)
    # 1. Rotación alrededor del eje Z por -Omega (Longitud del Nodo Ascendente)
    Rz_minus_Omega = spy.rotate(-node, 3)
    # 2. Rotación alrededor del eje X por -i (Inclinación)
    Rx_minus_i = spy.rotate(-i, 1)
    # 3. Rotación alrededor del eje Z por -w (Argumento del Periapsis)
    Rz_minus_peri = spy.rotate(-peri, 3)

    # Combinar las matrices para obtener la transformación de perifocal a astronómico
    M_perifocal2astro = Rz_minus_Omega @ Rx_minus_i @ Rz_minus_peri

    # Aplicar la transformación
    r_astro = M_perifocal2astro @ r_perifocal

    # Transponer para obtener un array de Nx3 (N puntos, 3 coordenadas)
    return r_astro.T

## Rutina para generar el visualizador

In [14]:
def create_asteroid_viewer(a, e, i, node, peri, name="Asteroide", grid_lim=1.5):
    """
    Genera y muestra un visualizador 3D para un cuerpo menor dado sus elementos orbitales,
    incluyendo la Tierra como referencia y marcas geométricas.
    """

    # Elementos orbitales de la Tierra (J2000)
    # Moved inside the function as requested
    deg = np.pi/180
    a_earth = 1.0000010178
    e_earth = 0.0167086
    i_earth = 0.00005 * deg # Casi cero respecto a la eclíptica
    node_earth = -11.26064 * deg
    peri_earth = 102.94719 * deg

    # 1. Generar trayectorias
    orbit_body = orbital_elements_to_orbit_3d(
        e=e, a=a, i=i,
        node=node, peri=peri, num_points=400
    )

    # Usamos los elementos de la Tierra definidos previamente
    orbit_earth = orbital_elements_to_orbit_3d(
        e=e_earth, a=a_earth, i=i_earth,
        node=node_earth, peri=peri_earth, num_points=400
    )

    fig = go.Figure()

    # --- Sol ---
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                               marker=dict(size=10, color='yellow'), name='Sol'))

    # --- Dirección Vernal (X) ---
    fig.add_trace(go.Scatter3d(x=[0, grid_lim*0.8], y=[0, 0], z=[0, 0],
                               mode='lines+text', text=['', '♈ Vernal'],
                               line=dict(color='magenta', width=4), name='Punto Vernal'))

    # --- Órbita de la Tierra ---
    fig.add_trace(go.Scatter3d(x=orbit_earth[:, 0], y=orbit_earth[:, 1], z=orbit_earth[:, 2],
                               mode='lines', line=dict(color='#40a0ff', width=2, dash='dot'), name='Órbita Tierra'))

    # --- Órbita del Objeto ---
    fig.add_trace(go.Scatter3d(x=orbit_body[:, 0], y=orbit_body[:, 1], z=orbit_body[:, 2],
                               mode='lines', line=dict(color='white', width=4), name=f'Órbita {name}'))

    # --- Picket-fence (Líneas a la eclíptica) ---
    for p in orbit_body[::12]:
        fig.add_trace(go.Scatter3d(x=[p[0], p[0]], y=[p[1], p[1]], z=[0, p[2]],
                                   mode='lines', line=dict(color='gray', width=1), opacity=0.3, showlegend=False))

    # --- Línea de Nodos ---
    # Convert 'node' (in radians) to degrees for use with np.radians if necessary, but 'node' is already in radians.
    raan_rad = node # node is already in radians from input
    node_vec = np.array([np.cos(raan_rad), np.sin(raan_rad), 0]) * (a * 1.3)
    fig.add_trace(go.Scatter3d(
        x=[-node_vec[0], node_vec[0]], y=[-node_vec[1], node_vec[1]], z=[0, 0],
        mode='lines+text', text=['☋ Desc.', '☊ Asc.'],
        line=dict(color='#00ffff', width=5), name='Línea de Nodos'
    ))

    # --- Eje de Ápsides ---
    peri_pos = orbit_body[0]
    apo_pos = orbit_body[len(orbit_body)//2]
    fig.add_trace(go.Scatter3d(x=[peri_pos[0], apo_pos[0]], y=[peri_pos[1], apo_pos[1]], z=[peri_pos[2], apo_pos[2]],
                               mode='lines', line=dict(color='red', width=2), name='Eje de Ápsides'))
    fig.add_trace(go.Scatter3d(x=[peri_pos[0], apo_pos[0]], y=[peri_pos[1], apo_pos[1]], z=[peri_pos[2], apo_pos[2]],
                               mode='markers+text', text=['Peri', 'Apo'],
                               marker=dict(color='red', size=4), showlegend=False))

    # --- Plano Eclíptico ---
    x_grid, y_grid = np.meshgrid(np.linspace(-grid_lim, grid_lim, 5), np.linspace(-grid_lim, grid_lim, 5))
    fig.add_trace(go.Surface(x=x_grid, y=y_grid, z=np.zeros_like(x_grid),
                             opacity=0.1, showscale=False, colorscale=[[0, 'white'], [1, 'white']], name='Eclíptica'))

    fig.update_layout(
        title=f"Orbit Viewer Geométrico: {name}",
        scene=dict(xaxis_title='X (AU)', yaxis_title='Y (AU)', zaxis_title='Z (AU)',
                   aspectmode='data', bgcolor='black'),
        template="plotly_dark",
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

## Ejemplo de prueba: Apophis

In [15]:
# Elementos orbitales de Apophis (valores aproximados y de ejemplo)

a_apophis = 0.9223803173917017 # AU
e_apophis = 0.1911663355386932
i_apophis = 3.340958441017069 * deg # radianes
node_apophis = 203.8996515621043 * deg # radianes
peri_apophis = 126.6728325163065 * deg # radianes

print(f"Elementos orbitales de Apophis (ejemplo):")
print(f"  Semieje mayor (a):     {a_apophis:.6f} AU")
print(f"  Excentricidad (e):     {e_apophis:.6f}")
print(f"  Inclinación (i):       {i_apophis/deg:.2f} grados ({i_apophis:.4f} rad)")
print(f"  Long. Nodo Asc. (node): {node_apophis/deg:.2f} grados ({node_apophis:.4f} rad)")
print(f"  Arg. Periapsis (peri): {peri_apophis/deg:.2f} grados ({peri_apophis:.4f} rad)")


Elementos orbitales de Apophis (ejemplo):
  Semieje mayor (a):     0.922380 AU
  Excentricidad (e):     0.191166
  Inclinación (i):       3.34 grados (0.0583 rad)
  Long. Nodo Asc. (node): 203.90 grados (3.5587 rad)
  Arg. Periapsis (peri): 126.67 grados (2.2109 rad)


In [16]:
create_asteroid_viewer(
    a=a_apophis,
    e=e_apophis,
    i=i_apophis,
    node=node_apophis,
    peri=peri_apophis,
    name="Apophis"
)